In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

c:\Users\Person\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Person\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
load_dotenv()

True

In [5]:
model = ChatGroq(model='openai/gpt-oss-120b')

In [6]:
# define state
class LLMState(TypedDict):
    question: str
    answer: str

In [7]:
def llm_qa(state: LLMState) -> LLMState:
    # extract question from state
    question = state['question']
    
    # form prompt for model
    prompt = f'Answer the following question: {question}'
    
    # get answer from model
    answer = model.invoke(prompt)
    
    # update the answer in the state
    state['answer'] = answer
    
    return state
    

In [8]:
graph = StateGraph(LLMState)

# add node
graph.add_node('llm_qa',llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile the graph
workflow = graph.compile()

In [11]:
# execute the graph
initial_state = {'question':'How far is the moon from the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state['answer'])

content='The Moon’s distance from Earth isn’t a single fixed number because its orbit is slightly elliptical.\u202fTypical values are:\n\n| Parameter | Approximate distance |\n|-----------|----------------------|\n| **Average (mean) distance** | **384\u202f400\u202fkm** (≈\u202f238\u202f900\u202fmi) |\n| **Perigee (closest approach)** | ~363\u202f300\u202fkm (≈\u202f225\u202f600\u202fmi) |\n| **Apogee (farthest point)** | ~405\u202f500\u202fkm (≈\u202f251\u202f900\u202fmi) |\n\nSo, on average the Moon is about **384\u202f000\u202fkilometers (≈\u202f239\u202f000\u202fmiles)** away from Earth, ranging roughly from **363\u202f000\u202fkm** at its nearest to **405\u202f000\u202fkm** at its farthest.' additional_kwargs={'reasoning_content': 'The user asks: "How far is the moon from the earth?" Straightforward factual answer. Provide average distance ~384,400 km, range 363,300 km (perigee) to 405,500 km (apogee). Could also give in miles. Provide context.'} response_metadata={'token_usage': 